# Network Flow V2 — Step 04: Pre-split compatibility and time-window readiness

**Purpose:** Verify the most promising **ToN-IoT ⇄ X-IIoTID** network-feature candidates, and audit timestamp representations in Edge-IIoTset and X-IIoTID before any 70/15/15 split.

**Important:** This notebook is **read-only on your raw CSVs**. It **does not** clean or merge files, train models, or make train/validation/test partitions. Its output is diagnostic evidence, **not** proof of cross-dataset semantic equivalence.

**Terms:**
- *Feature harmonization*: deciding whether columns in different datasets really measure the same thing (including direction, units, time basis, and aggregation).
- *Placeholder*: a token like `-` that means unavailable or not applicable, not necessarily zero.
- *Window*: a timestamp-grounded summary of events from a device or flow over 5, 10, 15 or 20 seconds. A CSV's row order is **not** a verified clock.
- *Data leakage*: using labels, future information, or near-identical observations across training and held-out partitions, leading to misleading evaluation.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
RAW = ROOT / "data" / "raw"
OUT = ROOT / "reports" / "audits"
DOCS = ROOT / "docs"
OUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

FILES = {
    "edge_iiotset": RAW / "edge_iiotset" / "ML-EdgeIIoT-dataset.csv",
    "x_iiotid": RAW / "x_iiotid" / "X-IIoTID dataset.csv",
    "ton_iot": RAW / "ton_iot" / "ton_iot_train_test_network.csv",
}
for name, path in FILES.items():
    print(name, "FOUND" if path.is_file() else "MISSING", path)
assert all(path.is_file() for path in FILES.values()), "One or more raw CSV files are absent."


## A. Candidate cross-dataset flow features

These are **column-name-based candidates** only. Compare direction, byte meanings, durations and measurement granularity against the release documentation before deciding which to use. Edge-IIoTset's packet/protocol fields **must not** be silently treated as completed flows.

In [ ]:
FLOW_PAIRS = {
    "source_port": ("src_port", "Scr_port"),
    "destination_port": ("dst_port", "Des_port"),
    "duration": ("duration", "Duration"),
    "source_payload_bytes": ("src_bytes", "Scr_bytes"),
    "destination_payload_bytes": ("dst_bytes", "Des_bytes"),
    "missed_bytes": ("missed_bytes", "missed_bytes"),
    "source_packets": ("src_pkts", "Scr_pkts"),
    "destination_packets": ("dst_pkts", "Des_pkts"),
    "source_ip_bytes": ("src_ip_bytes", "Scr_ip_bytes"),
    "destination_ip_bytes": ("dst_ip_bytes", "Des_ip_bytes"),
}
CATEGORY_PAIRS = {
    "protocol": ("proto", "Protocol"),
    "service": ("service", "Service"),
    "connection_state": ("conn_state", "Conn_state"),
}
MAP = [
    {"concept": k, "ton_column": pair[0], "x_column": pair[1],
     "semantic_match_verified": "NO", "direction_units_verified": "NO",
     "gateway_extractable_verified": "NO"}
    for k, pair in (FLOW_PAIRS | CATEGORY_PAIRS).items()
]
display(pd.DataFrame(MAP))


## B. Whole-file, memory-conscious quality inspection

Read **selected columns only**, in chunks. For each candidate numerical column, report placeholder counts, values that cannot be converted to numbers, zeros, and observed ranges. Report categorical frequencies separately. This scans the full two CSV files rather than relying on the potentially class-ordered first 5,000 rows.

**Caution:** Similar distributions or names still do not establish equivalent definitions.

In [ ]:
MISSING_MARKERS = {"", "-", "?", "null", "none", "nan", "n/a", "na", "inf", "+inf", "-inf"}
LABELS = {"ton_iot": "label", "x_iiotid": "class3"}
COLUMN_LOOKUP = {
    "ton_iot": {k: v[0] for k, v in (FLOW_PAIRS | CATEGORY_PAIRS).items()},
    "x_iiotid": {k: v[1] for k, v in (FLOW_PAIRS | CATEGORY_PAIRS).items()},
}

def inspect_flow_csv(dataset, chunk_size=20000):
    mapping = COLUMN_LOOKUP[dataset]
    cols = list(set(mapping.values()) | {LABELS[dataset]})
    seen = 0
    totals = {k: Counter() for k in mapping}
    cats = {k: Counter() for k in CATEGORY_PAIRS}
    example_values = {k: [] for k in FLOW_PAIRS}

    for chunk in pd.read_csv(FILES[dataset], usecols=cols, dtype=str,
                             keep_default_na=False, chunksize=chunk_size,
                             low_memory=False):
        seen += len(chunk)
        for concept, column in mapping.items():
            token = chunk[column].astype(str).str.strip()
            placeholder = token.str.lower().isin(MISSING_MARKERS)
            stats = totals[concept]
            stats["rows"] += len(token)
            stats["placeholders"] += int(placeholder.sum())
            if concept in FLOW_PAIRS:
                number = pd.to_numeric(token.mask(placeholder), errors="coerce")
                valid = number.notna()
                stats["valid_numeric"] += int(valid.sum())
                stats["invalid_nonplaceholder"] += int((~placeholder & ~valid).sum())
                stats["numeric_zeros"] += int((number == 0).sum())
                stats["numeric_negatives"] += int((number < 0).sum())
                available = number[valid]
                if not available.empty:
                    stats["min"] = min(stats.get("min", float("inf")), float(available.min()))
                    stats["max"] = max(stats.get("max", float("-inf")), float(available.max()))
                if len(example_values[concept]) < 3:
                    values = token[(~placeholder) & valid].head(3).tolist()
                    example_values[concept] = (example_values[concept] + values)[:3]
            else:
                cats[concept].update(token.loc[~placeholder].str.lower().value_counts().to_dict())
        if seen % 200000 < chunk_size:
            print(dataset, f"{seen:,} rows scanned")

    output_rows = []
    for concept, col in mapping.items():
        s = totals[concept]
        row = {
            "dataset": dataset, "concept": concept, "column": col,
            "rows": s["rows"], "placeholder_n": s["placeholders"],
            "placeholder_pct": round(100*s["placeholders"]/s["rows"], 3) if s["rows"] else None,
            "verified_equivalent": False,
        }
        if concept in FLOW_PAIRS:
            row.update({
                "numeric_valid_n": s["valid_numeric"],
                "invalid_nonplaceholder_n": s["invalid_nonplaceholder"],
                "zero_n": s["numeric_zeros"],
                "negative_n": s["numeric_negatives"],
                "observed_min": s.get("min"), "observed_max": s.get("max"),
                "numeric_examples": ", ".join(example_values[concept]),
            })
        output_rows.append(row)
    return output_rows, {
        c: dict(counter.most_common(12)) for c, counter in cats.items()
    }

quality_rows = []
category_summaries = {}
for dataset in ("ton_iot", "x_iiotid"):
    rows, cat = inspect_flow_csv(dataset)
    quality_rows.extend(rows)
    category_summaries[dataset] = cat

quality = pd.DataFrame(quality_rows)
display(quality[["dataset", "concept", "column", "placeholder_pct",
                 "invalid_nonplaceholder_n", "zero_n", "observed_min", "observed_max"]]
        .sort_values(["concept", "dataset"]).reset_index(drop=True))

print("Category distribution top values (review spelling before encoding):")
for dataset, cat in category_summaries.items():
    print("\n", dataset)
    for concept, counts in cat.items():
        print(concept, counts)


## C. Timestamp readiness for 5 / 10 / 15 / 20-second windows

- **ToN-IoT downloaded network CSV:** Has no timestamp field. A row number is not a second. **No genuine temporal windows from this file alone.**
- **X-IIoTID:** `Timestamp` may be an epoch-seconds timestamp, but check missing values, ranges, device/session association and ordering.
- **Edge-IIoTset:** `frame.time` contains apparent string-formatted dates *and* tokens such as `0.0` and `6.0`; these must not be assumed to be valid capture times.

The summaries below are diagnostic; they **do not** construct windows or establish that records are temporally joinable.

In [ ]:
def summarize_timestamps(dataset, timestamp_column, label_column, date_column=None):
    columns = [timestamp_column, label_column]
    if date_column:
        columns.append(date_column)
    rows = 0
    missing = 0
    numeric = 0
    yearlike = 0
    placeholder = 0
    examples = {"numeric": [], "yearlike": [], "other": []}
    min_epoch = None
    max_epoch = None
    plausible_epoch = 0
    date_missing = 0

    for chunk in pd.read_csv(FILES[dataset], usecols=columns, dtype=str,
                             keep_default_na=False, chunksize=20000,
                             low_memory=False):
        values = chunk[timestamp_column].astype(str).str.strip()
        v_lower = values.str.lower()
        is_placeholder = v_lower.isin(MISSING_MARKERS)
        nums = pd.to_numeric(values.mask(is_placeholder), errors="coerce")
        is_num = nums.notna()
        is_yearlike = values.str.contains(r"(?:19|20)\d{2}", regex=True, na=False) & ~is_num
        rows += len(values)
        placeholder += int(is_placeholder.sum())
        numeric += int(is_num.sum())
        yearlike += int(is_yearlike.sum())
        missing += int(is_placeholder.sum())
        if date_column:
            date_missing += int(chunk[date_column].astype(str).str.strip().str.lower().isin(MISSING_MARKERS).sum())
        # 2000-01-01 to 2035-01-01, in epoch SECONDS only as a *plausibility* gate
        plausible = is_num & nums.between(946684800, 2051222400)
        plausible_epoch += int(plausible.sum())
        if plausible.any():
            mn, mx = float(nums[plausible].min()), float(nums[plausible].max())
            min_epoch = mn if min_epoch is None else min(min_epoch, mn)
            max_epoch = mx if max_epoch is None else max(max_epoch, mx)
        for key, mask in (("numeric", is_num), ("yearlike", is_yearlike),
                          ("other", ~(is_num | is_yearlike | is_placeholder))):
            if len(examples[key]) < 6:
                examples[key] = (examples[key] + values[mask].head(6).tolist())[:6]

    return {
        "dataset": dataset, "column": timestamp_column, "rows": rows,
        "placeholder_rows": placeholder, "numeric_token_rows": numeric,
        "yearlike_string_rows": yearlike, "plausible_epoch_seconds_rows": plausible_epoch,
        "epoch_seconds_min": min_epoch, "epoch_seconds_max": max_epoch,
        "date_field_placeholder_rows": date_missing if date_column else None,
        "examples": examples,
        "temporal_window_ready": False,
        "reason": "Unverified timestamp semantics, capture provenance and grouping",
    }

timestamp_reports = [
    summarize_timestamps("edge_iiotset", "frame.time", "Attack_label"),
    summarize_timestamps("x_iiotid", "Timestamp", "class3", "Date"),
]
print("ToN-IoT: current file has no timestamp column; windows NOT supported.")
for report in timestamp_reports:
    print("\n", report["dataset"])
    print({k:v for k,v in report.items() if k!="examples"})
    print("Examples:", report["examples"])
    if report["epoch_seconds_min"] is not None:
        print("Candidate epoch range (NOT VERIFIED):",
              pd.to_datetime(report["epoch_seconds_min"], unit="s", utc=True),
              "to", pd.to_datetime(report["epoch_seconds_max"], unit="s", utc=True))


## D. Save the pre-split decision evidence

This saves reports, not modified datasets. Verify the output and feature semantics against original papers / dataset documentation before approving a training schema.

**Split gate:** Hold the 70/15/15 split until the row grouping / leakage policy is decided. Later fit imputers, encoders, scalers and feature selection on **training only** and reserve the final test partition.

**Attack-type gate:** The label taxonomies differ across all three sources. Keep binary `Normal/Attack` and each dataset's multiclass scheme separate unless you document an evidence-based mapping.

In [ ]:
flow_quality_path = OUT / "ton_x_flow_candidate_quality_v1.csv"
gate_path = OUT / "three_dataset_pre_split_gate_v1.json"
md_path = DOCS / "THREE_DATASET_PRE_SPLIT_GATE_V1.md"
quality.to_csv(flow_quality_path, index=False)

report = {
    "purpose": "Candidate feature and time-window readiness; no training performed",
    "candidate_pairs": MAP,
    "numeric_quality": quality.where(pd.notna(quality), None).to_dict(orient="records"),
    "category_top_counts": category_summaries,
    "timestamp_diagnostics": timestamp_reports,
    "dataset_split_status": "NOT_FINALIZED",
    "dataset_temporal_window_support": {
        "edge_iiotset": "UNVERIFIED (mixed representation)",
        "x_iiotid": "UNVERIFIED (timestamp exists, session alignment not verified)",
        "ton_iot": "NOT_SUPPORTED by current timestamp-free CSV"
    },
    "training_input_labels": {
        "edge_iiotset": ["Attack_label", "Attack_type"],
        "x_iiotid": ["class1", "class2", "class3"],
        "ton_iot": ["label", "type"],
    },
}
with gate_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str, ensure_ascii=False)

md_text = """# Three-Dataset Pre-Split Compatibility Gate (V1)

## Current decision
- **Edge-IIoTset:** train a dataset-specific packet/protocol baseline; do NOT assume its fields are pre-aggregated flows.
- **X-IIoTID + ToN-IoT:** promising *candidate* flow statistics; semantics, units, directions, placeholder handling and live gateway feasibility require source validation.
- **Time windows (5/10/15/20 s):** NOT approved on any file yet; no genuine event timestamps in the downloaded ToN-IoT CSV; inspect mixed `frame.time` in Edge-IIoTset; verify capture meaning for X-IIoTID.
- **Splits:** plan 70/15/15 within each dataset, but group/session/time-aware logic and final source provenance are prerequisites. Do not train preprocessing on validation/test partitions.
- **Models:** binary Normal/Attack first; multiclass labels differ between datasets and require separate schemas.

## What this does not establish
Column name similarity does not imply equivalent feature meaning. File order does not establish chronology. A dataset's attack class is not a proven complete APT campaign.

## Saved diagnostic outputs
- `reports/audits/ton_x_flow_candidate_quality_v1.csv`
- `reports/audits/three_dataset_pre_split_gate_v1.json`

## Manual verification actions
1. Review source documentation for each candidate feature (direction, measurement unit, packet vs flow semantics).
2. Verify whether group/session/capture identifiers and timestamp ordering can be reconstructed.
3. Investigate placeholder values `-` and missingness per attack category; do not convert them all to zero.
4. Freeze a leakage-aware split policy; use train-only encoders/imputers/scalers.
5. Reserve binary/multiclass benchmarks and temporal-window experiments as distinct studies until they can be linked with verified evidence.
"""
md_path.write_text(md_text, encoding="utf-8")
print("Saved", flow_quality_path)
print("Saved", gate_path)
print("Saved", md_path)


## What to send back

Copy **(1)** the candidate quality table (especially Duration/bytes/packets placeholder percentages and ranges), **(2)** the `frame.time` and `Timestamp` diagnostic summaries and examples, and **(3)** any warnings/errors. Then we can freeze the first defensible model-input feature sets and plan the split.

**Do not run the existing Edge-IIoTset Step 03 provisional split as the final dataset split yet.**